# Research: MacroFactorRotation-QC

Research notebook for AIStocksBondsRotationAlgorithm

**Calibration target**: `AIStocksBondsRotationAlgorithm` in `main.py`

In [1]:
# Initialize QuantBook
from datetime import datetime
from QuantConnect.Research import QuantBook

qb = QuantBook()
symbols = {}

symbols['SPY'] = qb.add_equity('SPY', Resolution.DAILY).symbol
symbols['TLT'] = qb.add_equity('TLT', Resolution.DAILY).symbol
symbols['GLD'] = qb.add_equity('GLD', Resolution.DAILY).symbol

start = datetime(2020, 1, 1)
end = datetime(2026, 1, 1)
history = qb.history([symbols['SPY'], symbols['TLT'], symbols['GLD']], start, end, Resolution.DAILY)
print(f"History: {len(history)} rows")
if len(history) > 0:
    display(history.head())

History: 1508 rows


close        high         low        open  \
symbol time                                                                  
SPY    2020-01-02 16:00:00  317.873535  317.873535  315.588822  316.611316   
       2020-01-03 16:00:00  315.466514  316.670024  314.194511  314.272788   
       2020-01-06 16:00:00  316.670024  316.758086  313.460663  313.538940   
       2020-01-07 16:00:00  315.779622  316.567286  315.300175  316.063377   
       2020-01-08 16:00:00  317.462580  318.763937  315.730699  316.034023   

                                volume  
symbol time                             
SPY    2020-01-02 16:00:00  49989106.0  
       2020-01-03 16:00:00  62118801.0  
       2020-01-06 16:00:00  44723087.0  
       2020-01-07 16:00:00  35987134.0  
       2020-01-08 16:00:00  58998805.0

## 1. Data Exploration

Statistical overview: returns distribution, correlations, volatility.

In [2]:
import pandas as pd
import numpy as np

if len(history) == 0:
    print('WARNING: No history data available')
    closes = pd.DataFrame()
    returns = pd.DataFrame()
else:
    if isinstance(history.index, pd.MultiIndex):
        closes = history['close'].unstack(level=0)
    else:
        closes = history['close']
    returns = closes.pct_change().dropna()
    print("=== Returns Summary ===")
    print(returns.describe().round(4))
    print()
    print("=== Correlation Matrix ===")
    print(returns.corr().round(3))

=== Returns Summary ===
symbol        SPY
count   1507.0000
mean       0.0002
std        0.0088
min       -0.1094
25%        0.0000
50%        0.0000
75%        0.0000
max        0.0906

=== Correlation Matrix ===
symbol  SPY
symbol     
SPY     1.0


## 2. Signal Analysis

Evaluate trend-following indicators (EMA crosses, SMA filters).

In [3]:
# Signal computation
if len(returns) == 0:
    print('No data for signal analysis')
    signal = pd.Series(dtype=float)
else:
    # Compute trend signals
    spy_close = closes.iloc[:, 0] if closes.shape[1] > 0 else closes
    ema_50 = spy_close.ewm(span=50).mean()
    ema_200 = spy_close.ewm(span=200).mean()
    trend_signal = (ema_50 - ema_200) / ema_200
    signal = trend_signal.dropna()
    signal.name = 'ema_cross_signal'

print("=== Signal Statistics ===")
if len(signal) > 0:
    print(f"Signal mean: {signal.mean():.6f}")
    print(f"Signal std: {signal.std():.6f}")
    print(f"Signal range: [{signal.min():.6f}, {signal.max():.6f}]")

=== Signal Statistics ===
Signal mean: 0.011959
Signal std: 0.025296
Signal range: [-0.056400, 0.080823]


## 3. Parameter Sensitivity

Test different parameter values for max_bitcoin_weight, lookback_years.

In [4]:
# Parameter sensitivity scan
if len(returns) == 0:
    print('No data for parameter scan')
    results = []
    best_params = 'N/A'
else:
    # Parameter scan for max_bitcoin_weight
    scan_values = [5, 10, 20, 30, 60]
    results = []
    for val in scan_values:
        # Compute metric for each parameter value
        vol = closes.iloc[:, 0].pct_change().rolling(val).std().mean()
        results.append((f'max_bitcoin_weight={val}', vol))
    best_params = min(results, key=lambda x: x[1])[0]

print("=== Parameter Scan Results ===")
for params, metric in results:
    print(f"  {params}: metric={metric:.4f}")
print(f"\nBest: {best_params}")

=== Parameter Scan Results ===
  max_bitcoin_weight=5: metric=0.0030
  max_bitcoin_weight=10: metric=0.0031
  max_bitcoin_weight=20: metric=0.0032
  max_bitcoin_weight=30: metric=0.0033
  max_bitcoin_weight=60: metric=0.0032

Best: max_bitcoin_weight=5


## 4. Calibration to main.py

Mapping research findings to algorithm parameters:

| Parameter | Research Value | main.py Default |
|-----------|---------------|----------------|
| `max_bitcoin_weight` | (research value) | default |
| `lookback_years` | (research value) | default |

In [5]:
# Calibration summary
calibration = {
    "max_bitcoin_weight": "<research_value>",
    "lookback_years": "<research_value>"
}

print("=== Calibration Parameters ===")
for k, v in calibration.items():
    print(f"  {k}: {v}")
print("\nTo apply: update get_parameter() defaults in main.py")

=== Calibration Parameters ===
  max_bitcoin_weight: <research_value>
  lookback_years: <research_value>

To apply: update get_parameter() defaults in main.py


## 5. Conclusion & Next Iteration

**Findings:**
- Universe (SPY, TLT, GLD) analyzed with trend_following signals
- Parameter sensitivity for max_bitcoin_weight, lookback_years scanned

**Next iteration:**
- Test alternative universe composition
- Add regime-conditional analysis
- Validate against backtest results in main.py